# XGBOOST

El modelo debe predecir a los aprobados ya que con esta información, la universidad diseñará los programas de becas por el curso. El Recall es importante porque te ayudará a evitar que el modelo se pierda estudiantes que merecían ser considerados para las becas, lo cual afectaría el diseño del programa.(evitar falsos negativos).

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score

from xgboost import XGBClassifier

data = pd.read_csv('/content/drive/MyDrive/TESIS/dataset_tdah_completo.csv', sep=';')

data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TESIS/dataset_tdah_completo.csv'

In [ ]:
data.shape

(500, 40)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data['tdah_presente'].value_counts()

,count
tdah_presente,
0,343
1,157


In [ ]:
columnas_eliminar = [
    'id',
    'tdah_presente',
    'tdah_probabilidad',
    'split',

    'edah_h1','edah_h2','edah_h3','edah_h4','edah_h5',
    'edah_da1','edah_da2','edah_da3','edah_da4','edah_da5',

    'edah_tc1','edah_tc2','edah_tc3','edah_tc4','edah_tc5',
    'edah_tc6','edah_tc7','edah_tc8','edah_tc9','edah_tc10',

    'edah_h_total',
    'edah_da_total',
    'edah_tdah_total',
    'edah_tc_total',

    'conners_hiperact_tscore',
    'conners_inatencion_tscore',
    'conners_oposicion_tscore',
    'conners_adhd_index_tscore',

    'prom_atencion',
    'prom_hiperactividad'
]

X = data.drop(columns=columnas_eliminar)

y = data['tdah_presente']

X = pd.get_dummies(X, drop_first=True)

In [ ]:
X = data.drop(columns=columnas_eliminar)

y = data['tdah_presente']

In [ ]:
X = pd.get_dummies(
    X,
    columns=['genero', 'nivel_riesgo_academico'],
    drop_first=True
)

In [ ]:
print(X.shape)
X.head()

(500, 7)


,edad,grado,condicion_social,promedio_notas,genero_M,nivel_riesgo_academico_Bajo,nivel_riesgo_academico_Medio
0,12,7,0,10.2,True,False,True
1,14,9,1,18.8,False,True,False
2,13,8,0,18.3,True,True,False
3,12,7,0,12.7,False,True,False
4,14,9,3,10.0,True,False,True


In [ ]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(300, 7)
(100, 7)
(100, 7)


In [ ]:
from xgboost import XGBClassifier

xgb_0 = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    eval_metric='logloss',
    max_depth=4,
    n_estimators=100,
    learning_rate=0.05
)

xgb_0.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred_train = xgb_0.predict(X_train)
y_pred_val = xgb_0.predict(X_val)
y_pred_test = xgb_0.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix

print("TRAIN")
print(confusion_matrix(y_train, y_pred_train))

print("\nVALIDATION")
print(confusion_matrix(y_val, y_pred_val))

print("\nTEST")
print(confusion_matrix(y_test, y_pred_test))

TRAIN
[[205   1]
 [  2  92]]

VALIDATION
[[64  4]
 [ 6 26]]

TEST
[[63  6]
 [ 1 30]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.98      0.91      0.95        69
           1       0.83      0.97      0.90        31

    accuracy                           0.93       100
   macro avg       0.91      0.94      0.92       100
weighted avg       0.94      0.93      0.93       100



In [ ]:
from sklearn.metrics import confusion_matrix

print("TRAIN")
print(confusion_matrix(y_train, y_pred_train))

print("VALIDATION")
print(confusion_matrix(y_val, y_pred_val))

print("TEST")
print(confusion_matrix(y_test, y_pred_test))

TRAIN
[[205   1]
 [  2  92]]
VALIDATION
[[64  4]
 [ 6 26]]
TEST
[[63  6]
 [ 1 30]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.98      0.91      0.95        69
           1       0.83      0.97      0.90        31

    accuracy                           0.93       100
   macro avg       0.91      0.94      0.92       100
weighted avg       0.94      0.93      0.93       100



In [ ]:
# Separar características y target
X = data.drop('tdah_presente', axis=1)
y = data['tdah_presente']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score

In [ ]:
# Dividir el dataset
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

In [ ]:
X_train.shape, X_val.shape, X_test.shape , X_trainval.shape

((300, 39), (100, 39), (100, 39), (400, 39))

In [ ]:
from xgboost import XGBClassifier

## Modelo 0 - XGboost valores por defecto

### Modelar

In [ ]:
#Entrenamiento TRAIN
xgb_0 = XGBClassifier(
    random_state=42,
    objective= 'binary:logistic',
    enable_categorical=True # Habilitar el manejo de características categóricas (experimental)
    )

# Identificar las columnas de tipo 'object' en X_train y convertirlas a 'category'
categorical_cols = X_train.select_dtypes(include='object').columns
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')

#objective='multi:softmax',   # Para clasificación múltiple
#num_class=3,
xgb_0.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
#Ver estabilidad CV
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xgb_0, X_train, y_train, cv=5, scoring='recall')  # o 'f1', 'roc_auc', etc.

print("CV recall scores:", scores)
print("Mean CV recall:", scores.mean())
print("Std CV recall (estabilidad):", scores.std())

CV recall scores: [1.         0.94736842 1.         1.         1.        ]
Mean CV recall: 0.9894736842105264
Std CV recall (estabilidad): 0.02105263157894739


In [ ]:
# Generar las predicciones con TRAIN
y_pred_train = xgb_0.predict(X_train)

In [ ]:
# Generar las predicciones con VALID
# Identificar las columnas de tipo 'object' en X_val y convertirlas a 'category'
categorical_cols_val = X_val.select_dtypes(include='object').columns
for col in categorical_cols_val:
    X_val[col] = X_val[col].astype('category')

y_pred_val = xgb_0.predict(X_val)

### Métricas y visualizacion

In [ ]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [ ]:
print("Matrix confusion de TRAIN:")  ##OVERFITI
print(confusion_matrix(y_train, y_pred_train))

print("Matrix confusion de VAL:")
print(confusion_matrix(y_val, y_pred_val))

Matrix confusion de TRAIN:
[[206   0]
 [  0  94]]
Matrix confusion de VAL:
[[68  0]
 [ 0 32]]


In [ ]:
print("Métricas en TRAIN:")
print(classification_report(y_train, y_pred_train))

# Predicción en VALID
print("Métricas en VALID:")
print(classification_report(y_val, y_pred_val))

Métricas en TRAIN:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       206
           1       1.00      1.00      1.00        94

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

Métricas en VALID:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00        32

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [ ]:
#Hiperparámetros por defecto
default_params = xgb_0.get_params()
print(default_params)

{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}


## Modelo 1

#### Modelar

In [ ]:
xgb01 = XGBClassifier(
    objective      ='binary:logistic',   # Para clasificación binaria
    eval_metric    = ['logloss','auc'], # Métrica de monitoreo (cuando detenerse)## LOGLOOSS MIDE EL ERROR ###AUC----QUE TAN CERCA ESTÁ DE LA REALIDAD, MIENTRAS MAS CERCA A 1, ##LE HACE CASO AL AUC, SI SE INVIERTE HACE CASO  AL PRIMERO, USUALMENTE AL SEGVUNDO
    random_state=42,
    enable_categorical=True # Add this line to enable categorical feature handling
    #scale_pos_weight=2.7
    )

In [ ]:
# Definir la cuadrícula de hiperparámetros
# param_grid = {
#     'n_estimators': np.arange(50, 300, 50),
#     'max_depth': np.arange(3, 10),
#     'learning_rate': [0.01, 0.1, 0.2], #()np.linspace(0.01, 0.1, 0.2, 0.3, 10),
#     'subsample': np.linspace(0.6, 1.0, 5),
#     'colsample_bytree': np.linspace(0.6, 1.0, 5)
# }


# param_grid = {
#     'n_estimators': np.arange(100, 500, 50),  # Rango más amplio y flexible
#     'max_depth': [3, 4, 5, 6, 7, 8],          # No muy grandes para evitar overfitting
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Más granularidad para aprendizaje
#     'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],   # Evita overfitting si es < 1
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],  # Igual que subsample
#     'gamma': [0, 0.1, 0.3, 0.5, 1],           # Regularización para poda de ramas débiles
#     'reg_alpha': [0, 0.1, 0.5, 1],            # L1 regularization (sparse)
#     'reg_lambda': [0.5, 1, 1.5, 2]            # L2 regularization
# }


param_grid = {
    'n_estimators': np.arange(50, 300, 50),   ###CANTIDAD DE ARBOLES
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],  ##TAMAÑO DEL PASO,
    'subsample': np.linspace(0.6, 1.0, 4), ##FRACCION DE MUESTRA DE REGISTROS
    'colsample_bytree': np.linspace(0.6, 1.0, 4), ##FEACCIONES DE MUESRA  POR COLUMNA
    'reg_alpha': [0, 0.1, 1, 5],         # L1 regularización ###PENALIZACION DE LOS  ERRORES
    'reg_lambda': [1, 5, 10],            # L2 regularización ###PENALIZACIOND DE LOS ERRORES
    'min_child_weight': [1, 3, 5, 10],   # Controla el sobreajuste ###
}

In [ ]:
#Entrenar el modelo con los hiperparámrtros y los datos de TRAIN
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


random_search = RandomizedSearchCV(
        estimator  = xgb01,
        param_distributions = param_grid ,
        cv = skf,
        n_iter = 50, ##RAMDONIZD VA TOMAR SOLO 50
        n_jobs = -1,
        verbose = 8,
        scoring = 'recall',
        random_state = 42
      )


random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric=['logloss', 'auc'],
                                           feature_types=None,
                                           feature_weights=None, gam...
                   n_iter=50, n_jobs=-1,
                   param_distributions={'colsample_bytree': array([0.6       , 0.73333333, 0.86666667, 1.        ]),
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [3, 4, 5, 6],
                                        'min_child_weight': [1, 3, 5, 10],
                                        'n_estimators': array([ 50, 100, 150, 200, 250]),
                                        'reg_alpha': [0, 0.1, 1, 5],
                                        'reg_lambda': [1, 5, 10],
                                        'subsample': array([0.6       , 0.73333333, 0.86666667, 1.        ])},
                   random_state=42, scoring='recall', verbose=8)

In [ ]:
#Ver la estabilidad del modelo (promedio y desviacion) y los mejores hiperparámetros
import pandas as pd
cv_results = pd.DataFrame(random_search.cv_results_)
cv_results.head(4)
#cv_results[['mean_test_score', 'std_test_score', 'params']]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_subsample,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,...,param_colsample_bytree,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.138736,0.033271,0.039159,0.010662,1.000000,1,0.0,250,3,3,...,0.733333,"{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1
1,0.138297,0.033534,0.027237,0.011953,0.600000,10,5.0,150,10,3,...,0.600000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,0.947368,1.0,1.0,0.978363,0.026516,47
2,0.263883,0.046051,0.029947,0.010485,0.866667,10,5.0,250,5,6,...,1.000000,"{'subsample': 0.8666666666666667, 'reg_lambda'...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1
3,0.136233,0.035911,0.034380,0.004770,0.600000,10,5.0,100,5,5,...,1.000000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1


In [ ]:
cv_results.sort_values(by='rank_test_score', ascending=True).head(4)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_subsample,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,...,param_colsample_bytree,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.138736,0.033271,0.039159,0.010662,1.000000,1,0.0,250,3,3,...,0.733333,"{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
2,0.263883,0.046051,0.029947,0.010485,0.866667,10,5.0,250,5,6,...,1.000000,"{'subsample': 0.8666666666666667, 'reg_lambda'...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
3,0.136233,0.035911,0.034380,0.004770,0.600000,10,5.0,100,5,5,...,1.000000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
4,0.091179,0.002827,0.037726,0.002677,0.600000,5,0.0,50,10,6,...,0.600000,"{'subsample': 0.6, 'reg_lambda': 5, 'reg_alpha...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1


In [ ]:
print("CV recall scores:", scores)  ##NP.INT64---250 ARBOLES CON EL MEJOR RESULTADO
print("Mejor recall promedio (CV):", random_search.best_score_)
print("Mejores hiperparámetros:", random_search.best_params_)
#xgb01_random_search =  random_search.best_estimator_
best_params = random_search.best_params_

CV recall scores: [1.         0.94736842 1.         1.         1.        ]
Mejor recall promedio (CV): 0.9888888888888889
Mejores hiperparámetros: {'subsample': np.float64(1.0), 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': np.int64(250), 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': np.float64(0.7333333333333333)}


In [ ]:



# Entrenamos con early stopping usando VALIDACIÓN EXTERNA (val) ##el auc busca el mas alto

xgb01 = XGBClassifier(
    **best_params,
    eval_metric           = ['auc','logloss'], # Métrica de monitoreo (cuando detenerse) USA LA SEGUNDA DE LA LISTA
    objective             = 'binary:logistic',##LOGLOSS OBTIENE DEL VAL Y EL AUC DEL TRAIN
    random_state          = 42,
    early_stopping_rounds = 10,
    enable_categorical=True # Explicitly add this parameter
    )

xgb01.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=True
)

#Guardar la mejor iteración
best_iter = xgb01.best_iteration
print("Mejor iteración:", best_iter)

[0]	validation_0-auc:1.00000	validation_0-logloss:0.57487	validation_1-auc:1.00000	validation_1-logloss:0.57957
[1]	validation_0-auc:1.00000	validation_0-logloss:0.53347	validation_1-auc:1.00000	validation_1-logloss:0.53772
[2]	validation_0-auc:1.00000	validation_0-logloss:0.49651	validation_1-auc:1.00000	validation_1-logloss:0.50037
[3]	validation_0-auc:1.00000	validation_0-logloss:0.46323	validation_1-auc:1.00000	validation_1-logloss:0.46676
[4]	validation_0-auc:1.00000	validation_0-logloss:0.43358	validation_1-auc:1.00000	validation_1-logloss:0.43650
[5]	validation_0-auc:1.00000	validation_0-logloss:0.40605	validation_1-auc:1.00000	validation_1-logloss:0.40872
[6]	validation_0-auc:1.00000	validation_0-logloss:0.38085	validation_1-auc:1.00000	validation_1-logloss:0.38331
[7]	validation_0-auc:1.00000	validation_0-logloss:0.35815	validation_1-auc:1.00000	validation_1-logloss:0.36014
[8]	validation_0-auc:1.00000	validation_0-logloss:0.33676	validation_1-auc:1.00000	validation_1-logloss:

In [ ]:
#Mejor score
print("Mejor score:", xgb01.best_score) # siempre la ultima metrica definida
print(min(xgb01.evals_result()['validation_1']['logloss'])) #si es el logloss se busca el menor

Mejor score: 0.02068707019090653
0.02068707019090653


In [ ]:
#Accesar a los valores
xgb01.evals_result() #['validation_1']['logloss']

{'validation_0': OrderedDict([('auc',
               [1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1

In [ ]:
# Generar las predicciones con TRAIN
y_pred_train = xgb01.predict(X_train)
# Generar las predicciones con VALID
y_pred_val = xgb01.predict(X_val)

#### Métricas y visualizacion

In [ ]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [ ]:
print("Matrix confusion de TRAIN:")
print(confusion_matrix(y_train, y_pred_train))

print("Matrix confusion de VAL:")
print(confusion_matrix(y_val, y_pred_val))

Matrix confusion de TRAIN:
[[206   0]
 [  0  94]]
Matrix confusion de VAL:
[[68  0]
 [ 0 32]]


In [ ]:
print("Métricas en TRAIN:")
print(classification_report(y_train, y_pred_train))


print("Métricas en VALID:")
print(classification_report(y_val, y_pred_val))

Métricas en TRAIN:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       206
           1       1.00      1.00      1.00        94

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

Métricas en VALID:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00        32

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



## Evaluar el modelo final

In [ ]:
params_sin_estimators = best_params.copy()
params_sin_estimators.pop('n_estimators', None)

final_model = XGBClassifier(
    **params_sin_estimators,
    n_estimators=best_iter + 1,  # # Para evitar overfitting
    random_state=42,
    enable_categorical=True  # Add this parameter to handle categorical features
)

# Convert 'object' columns in X_trainval to 'category' dtype
categorical_cols_trainval = X_trainval.select_dtypes(include='object').columns
for col in categorical_cols_trainval:
    X_trainval[col] = X_trainval[col].astype('category')

final_model.fit(X_trainval, y_trainval)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=np.float64(0.7333333333333333), device=None,
              early_stopping_rounds=None, enable_categorical=True,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=68, n_jobs=None,
              num_parallel_tree=None, ...)

#### Méricas con TEST

In [ ]:
# Convert 'object' columns in X_test to 'category' dtype
categorical_cols_test = X_test.select_dtypes(include='object').columns
for col in categorical_cols_test:
    X_test[col] = X_test[col].astype('category')

# Generar las predicciones con TEST
y_pred_test = final_model.predict(X_test)

In [ ]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [ ]:
print("Matrix confusion de TEST:")
print(confusion_matrix(y_test, y_pred_test))

Matrix confusion de TEST:
[[69  0]
 [ 0 31]]


In [ ]:
print("Métricas en TEST:")
print(classification_report(y_test, y_pred_test))

Métricas en TEST:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        69
           1       1.00      1.00      1.00        31

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, y_pred_test)

np.float64(1.0)

## Ejemplo de Predicción

In [ ]:
# Ejemplo de una persona a identificar
# The example_person DataFrame must have the same features as X_trainval
# Let's create a hypothetical person's data based on the features the model expects.
# Using X_trainval.columns to get the expected feature names.

# Create a dictionary for a hypothetical person with representative values
# We'll use the column names from X_trainval and fill with sample data.
# For simplicity, we can take a row from the original 'data' or X_trainval as a template.

# Get the feature names from X_trainval
expected_features = X_trainval.columns.tolist()

# Create a dictionary with sample data for a hypothetical person
# For numeric columns, provide sensible numbers.
# For categorical columns ('genero', 'nivel_riesgo_academico', 'split'), use string values that exist in your dataset.

# Example values (you might want to adjust these to represent a specific case)
# Let's take the mean for numerical features and mode for categorical features from the training data
example_data_dict = {}
for col in expected_features:
    # Check if the column is categorical (either 'object' or 'category' dtype)
    if X_trainval[col].dtype == 'object' or isinstance(X_trainval[col].dtype, pd.CategoricalDtype):
        example_data_dict[col] = [X_trainval[col].mode()[0]] # Use mode for categorical
    else:
        # Otherwise, it's numerical, use mean
        example_data_dict[col] = [X_trainval[col].mean()] # Use mean for numerical

# Manually adjust a few values to represent a person we want to predict for
# For instance, if 'edad' and 'promedio_notas' are key features
# Let's assume a 13-year-old male with average notes and academic risk.
example_data_dict['id'] = [999] # Placeholder ID
example_data_dict['edad'] = [13]
example_data_dict['genero'] = ['M']
example_data_dict['nivel_riesgo_academico'] = ['Bajo']
example_data_dict['promedio_notas'] = [15.0]
example_data_dict['split'] = ['train'] # This column was part of X and often used to split data, but for a single prediction, it might not be relevant to the model itself. However, it needs to be present.


example_person = pd.DataFrame(example_data_dict)

# Convert 'object' columns in example_person to 'category' dtype, similar to training data
categorical_cols_example = example_person.select_dtypes(include='object').columns
for col in categorical_cols_example:
    example_person[col] = example_person[col].astype('category')

# Realizar la predicción
prediction = final_model.predict(example_person)

# Mostrar la predicción
result = "Aprobado" if prediction[0] == 1 else "No Aprobado"
print(f"La persona con las características dadas será: {result}")

La persona con las características dadas será: No Aprobado


In [ ]:
# # Definir la cuadrícula de hiperparámetros
# param_grid = {
#     'n_estimators': [20, 30, 50],                   # Número de árboles
#     'learning_rate': [0.001, 0.01, 0.1, 0.2],                 # Tasa de aprendizaje
#     'max_depth': [3, 4, 5, 6, 7],                      # Profundidad máxima de un árbol
#     #'min_child_weight': [1, 2, 3, 4],                  # Mínimo peso de los hijos
#     'colsample_bytree': [0.6, 0.8, 1.0],               # Proporción de columnas utilizadas por árbol
#     'gamma': [0, 0.1, 0.2, 0.3],                        # Reducción de la función de pérdida
#     #'scale_pos_weight': [1, 2, 3],                     # Peso para clases desbalanceadas
#     'objective': ['binary:logistic'],                   # Función de objetivo
#     'eval_metric': ['logloss','auc', 'error'],                   # Métrica de evaluación
#     #'booster': ['gbtree', 'gblinear']                   # Tipo de booster
#     'reg_alpha': [0.01, 0.1, 1],  # L1 regularization
#     'reg_lambda': [0.01, 0.1, 1],  # L2 regularization
#     'subsample': [0.7, 0.8, 0.9]  # Para evitar que el modelo aprenda demasiado de un subconjunto
# }


# param_grid = {
#     'n_estimators': [50, 100, 200],               # Menos árboles
#     'learning_rate': [0.001,0.01, 0.05, 0.1],           # Tasas de aprendizaje más bajas
#     'max_depth': np.arange(5, 30, 2),                    # Profundidad de árbol más baja
#     'min_child_weight': [1, 2],             # Ponderación de hijos mínima
#     'colsample_bytree': [0.6, 0.8],         # Menos columnas por árbol
#     'gamma': [0, 0.1],                      # Control de pérdida
#     'reg_alpha': [0.1, 0.5, 1],                  # Aumento de regularización L1
#     'reg_lambda': [0.1, 0.5, 1],                 # Aumento de regularización L2
#     'subsample': [0.7, 0.8, 0.9],                # Fracción de datos por árbol
#     'objective': ['binary:logistic'],       # Función de objetivo
#     'eval_metric': ['logloss', 'auc'],      # Métricas de evaluación,
#     'scale_pos_weight' : [2]
# }

# # xgboost_model = xgb.XGBClassifier(
# #     n_estimators=100,               # Número de árboles
# #     max_depth=5,                    # Profundidad máxima del árbol
# #     learning_rate=0.1,              # Tasa de aprendizaje
# #     subsample=0.8,                  # Proporción de muestras a usar para cada árbol
# #     colsample_bytree=0.8,           # Proporción de características a usar para cada árbol
# #     gamma=0,                        # Complejidad de la penalización
# #     reg_alpha=0,                    # Regularización L1
# #     reg_lambda=1,                   # Regularización L2
# #     use_label_encoder=False,        # Usar encoder de etiquetas
# #     eval_metric='logloss'           # Métrica de evaluación
# # )